# HW2 - 2D Diffusion

Question: how is denoising different from flow?

Sources:

- DDPM: https://arxiv.org/abs/2006.11239
- Diffusion Explainer: https://poloclub.github.io/diffusion-explainer/
- Hugging Face Annotated Diffusion: https://github.com/huggingface/blog/blob/main/annotated-diffusion.md


In [ ]:
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(7)
random.seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
def sample_moons(n, noise=0.06):
    n1 = n // 2
    n2 = n - n1
    t1 = torch.rand(n1) * math.pi
    t2 = torch.rand(n2) * math.pi
    x1 = torch.stack([torch.cos(t1), torch.sin(t1)], dim=1)
    x2 = torch.stack([1 - torch.cos(t2), 0.5 - torch.sin(t2)], dim=1)
    x = torch.cat([x1, x2], dim=0)
    x = x + noise * torch.randn_like(x)
    return (x - x.mean(0)) / x.std(0)


T = 100
betas = torch.linspace(1e-4, 0.04, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)


def q_sample(x0, t, eps=None):
    if eps is None:
        eps = torch.randn_like(x0)
    ab = alpha_bars[t].view(-1, 1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps, eps


In [ ]:
x0 = sample_moons(2500).to(device)
timesteps = [0, 10, 30, 60, 99]
fig, axes = plt.subplots(1, len(timesteps), figsize=(15, 3))
for ax, ti in zip(axes, timesteps):
    t = torch.full((x0.shape[0],), ti, device=device, dtype=torch.long)
    xt, _ = q_sample(x0, t)
    pts = xt.cpu()
    ax.scatter(pts[:, 0], pts[:, 1], s=2)
    ax.set_title(f"t={ti}")
    ax.axis("equal")
    ax.axis("off")
plt.show()


In [ ]:
class DenoiseMLP(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, x, t):
        t_scaled = t.float()[:, None] / (T - 1)
        return self.net(torch.cat([x, t_scaled], dim=1))


model = DenoiseMLP().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)


In [ ]:
losses = []
for step in range(3500):
    x0 = sample_moons(512).to(device)
    t = torch.randint(0, T, (x0.shape[0],), device=device)
    xt, eps = q_sample(x0, t)
    pred_eps = model(xt, t)
    loss = F.mse_loss(pred_eps, eps)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if step % 500 == 0:
        print(step, round(loss.item(), 4))

plt.plot(losses)
plt.title("DDPM noise prediction loss")
plt.show()


In [ ]:
@torch.no_grad()
def sample_ddpm(n=2048):
    x = torch.randn(n, 2, device=device)
    snapshots = []
    for ti in reversed(range(T)):
        t = torch.full((n,), ti, device=device, dtype=torch.long)
        beta, alpha, ab = betas[ti], alphas[ti], alpha_bars[ti]
        pred_eps = model(x, t)
        mean = (1 / alpha.sqrt()) * (x - beta / (1 - ab).sqrt() * pred_eps)
        x = mean if ti == 0 else mean + beta.sqrt() * torch.randn_like(x)
        if ti in [99, 75, 50, 25, 0]:
            snapshots.append((ti, x.cpu()))
    return x.cpu(), snapshots


samples, snapshots = sample_ddpm()
fig, axes = plt.subplots(1, len(snapshots), figsize=(15, 3))
for ax, (ti, pts) in zip(axes, snapshots):
    ax.scatter(pts[:, 0], pts[:, 1], s=2)
    ax.set_title(f"t={ti}")
    ax.axis("equal")
    ax.axis("off")
plt.show()


## Diffusion vs Flow

| Topic | Flow matching | Diffusion |
| --- | --- | --- |
| Target | velocity | usually noise |
| Sampling | integrate learned vector field | reverse a noising chain |
| Easy visualization | trajectories | denoising snapshots |
| Failure mode | wrong motion field | bad noise estimate at some timesteps |

Interview line: diffusion is easier to introduce as "learn to remove noise"; flow matching is easier to view as "learn the transport direction."
